<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/fundamentos/notebooks/c1_l3.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C1-L3 · Volatilidad y colas
Calcula σ sobre retornos diarios de BTC, convierte cada día a sigmas y localiza el día de cola (+3σ).

In [ ]:
import pandas as pd
from pathlib import Path

CSV = 'c1_l3_btc_retornos.csv'
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/fundamentos/data/' + CSV
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data') / CSV, Path('data') / CSV, Path(CSV)]:
        if cand.exists():
            df = pd.read_csv(cand)
            break
    print('Fuente: local')
print(df.shape)
print(df.head())
print(df["retorno_pct"].describe().round(2))

## Sigma: la regla para medir sustos
La volatilidad es la desviación estándar de los retornos. Dividiendo cada día entre σ lo expresas en sigmas y puedes comparar sustos entre activos.

In [ ]:
media = df["retorno_pct"].mean()
sigma = df["retorno_pct"].std(ddof=0)
df["z"] = (df["retorno_pct"] - media) / sigma
print(f"media={media:+.2f}%  sigma={sigma:.2f}%")
extremo = df.loc[df["z"].abs().idxmax()]
print("día extremo:", extremo["fecha"], f'{extremo["retorno_pct"]:+.2f}%', f'z={extremo["z"]:+.2f}')
print("días más allá de ±2σ:", int((df["z"].abs() > 2).sum()))

## Histograma con la línea +3σ
La barra naranja de la lección aparece aquí como el valor más allá de la línea punteada: existe, y el resto se ve normal a su lado.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["retorno_pct"], bins=15, color="#5eead4", edgecolor="#0a0a0b")
ax.axvline(media + 3 * sigma, color="#f59e0b", linestyle="--", label="+3σ")
ax.axvline(media - 3 * sigma, color="#f59e0b", linestyle=":", label="−3σ")
ax.set_xlabel("retorno diario (%)")
ax.set_ylabel("días")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Chequeos automáticos
assert len(df) == 60, "se esperan 60 días"
assert sigma > 0
assert df["z"].abs().max() > 3, "el dataset incluye un día de cola (+3σ)"
print("OK: sigma calculada y día de cola localizado")